In [1]:
from gurobipy import *

In [6]:
#δεδομ΄ένα
warehouses = list(range(12))     # 0 εώς 11
distributors = list(range(12))

# συνολικά κόστη μεταφοράς, για άπειρο έθεσα 1e6 (τεράστια τιμή) οπότε θα αποφευχθεί
transport_cost = [
    [100, 80, 50, 50, 60, 100, 120, 90, 60, 70, 65, 110],
    [120, 90, 60, 70, 65, 110, 140, 110, 80, 80, 75, 130],
    [140, 110, 80, 80, 75, 130, 160, 125, 100, 100, 80, 150],
    [160, 125, 100, 100, 80, 150, 190, 150, 130, 1e6, 1e6, 1e6],
    [190, 150, 130, 1e6, 1e6, 1e6, 180, 150, 50, 50, 60, 100],
    [200, 180, 150, 1e6, 1e6, 1e6, 100, 120, 90, 60, 75, 110],
    [120, 90, 60, 70, 65, 110, 140, 110, 80, 80, 75, 130],
    [120, 90, 60, 70, 65, 110, 140, 110, 80, 80, 75, 130],
    [140, 110, 80, 80, 75, 130, 160, 125, 100, 100, 80, 150],
    [160, 125, 100, 100, 80, 150, 190, 150, 130, 1e6, 1e6, 1e6],
    [190, 150, 130, 1e6, 1e6, 1e6, 200, 180, 150, 1e6, 1e6, 1e6],
    [200, 180, 150, 1e6, 1e6, 1e6, 100, 80, 50, 50, 60, 100],
]

# πάγια κόστη
fixed_cost = [3500, 9000, 10000, 4000, 3000, 9000, 9000, 3000, 4000, 10000, 9000, 3500]

# χωρητικότητες
capacity = [300, 250, 100, 180, 275, 300, 200, 220, 270, 250, 230, 180]

# Ζήτηση κέντρων
demand = [120, 80, 75, 100, 110, 100, 90, 60, 30, 150, 95, 120]

# μοντέλο
m = Model("warehouse_location")

# Variables
# x[i,j] = πόσους τόνους στέλνει η αποθήκη i στον πελάτη j
x = m.addVars(warehouses, distributors, name="x", vtype=GRB.CONTINUOUS, lb=0)


# y[i] = 1 αν λειτουργήσει η αποθήκη i, αλλιώς 0
y = m.addVars(warehouses, name="y", vtype=GRB.BINARY)

# κόστος ανά unit όπως στην εκφώνηση για μεταφορά από αποθήκη σε distributor
cost_per_unit = {}
for i in warehouses:
    for j in distributors:
        if transport_cost[i][j] >= 1e6:
            cost_per_unit[i, j] = 1e6
        else:
            cost_per_unit[i, j] = transport_cost[i][j]/ demand[j] #δεν πολλαπλασίασα επί 1000 διότι θα βγάλω αποτέλεσμα σε cost = ...χιλ. ευρώ

#Η ζήτηση σε κάθε κέντρο πώλησης πρέπει να καλύπτεται, 
#ενώ το κάθε κέντρο μπορεί να εξυπηρετείται από πολλές αποθήκες.
for j in distributors:
    m.addConstr(quicksum(x[i,j] for i in warehouses) == demand[j])

#κάθε αποθήκη δεν μπορεί να στείλει πάνω από το capacity της και επίσης στέλνει μόνο εάν είναι ανοιχτή (y[i] == 1)
for i in warehouses:
    m.addConstr(quicksum(x[i, j] for j in distributors) <= capacity[i] * y[i])

#αντικειμενική συνάρτηση
m.setObjective(
    quicksum(fixed_cost[i] * y[i] for i in warehouses) + #άθροισμα πάγιων κόστων για το άνοιγμα των αποθηκ΄ων
    quicksum(cost_per_unit[i, j] * x[i, j] for i in warehouses for j in distributors), #άθροισμα κόστων μεταφοράς
    GRB.MINIMIZE
)

In [7]:
m.optimize()

Gurobi Optimizer version 12.0.2 build v12.0.2rc0 (win64 - Windows 11.0 (26100.2))

CPU model: AMD Ryzen 7 3700U with Radeon Vega Mobile Gfx, instruction set [SSE2|AVX|AVX2]
Thread count: 4 physical cores, 8 logical processors, using up to 8 threads

Optimize a model with 24 rows, 156 columns and 300 nonzeros
Model fingerprint: 0x6b1c0085
Variable types: 144 continuous, 12 integer (12 binary)
Coefficient statistics:
  Matrix range     [1e+00, 3e+02]
  Objective range  [3e-01, 1e+06]
  Bounds range     [1e+00, 1e+00]
  RHS range        [3e+01, 2e+02]
Found heuristic solution: objective 3.100484e+08
Presolve time: 0.02s
Presolved: 24 rows, 156 columns, 300 nonzeros
Variable types: 144 continuous, 12 integer (12 binary)

Root relaxation: objective 1.574011e+04, 37 iterations, 0.00 seconds (0.00 work units)

    Nodes    |    Current Node    |     Objective Bounds      |     Work
 Expl Unexpl |  Obj  Depth IntInf | Incumbent    BestBd   Gap | It/Node Time

     0     0 15740.1111    0    1 

In [10]:
#prints για debugging
if m.status == GRB.OPTIMAL:
    print("Βρέθηκε βέλτιστη λύση!")
else:
    print("Το πρόβλημα δεν λύθηκε σωστά.")

Βρέθηκε βέλτιστη λύση!


In [12]:
print("Ανοιχτές Αποθήκες:")
for i in warehouses:
    if y[i].X > 0.5:
        print(f" - Αποθήκη {i+1} (χωρητικότητα: {capacity[i]})") # i+1 διότι όρισα τους αρχικούς πίνακες range(12) = 0,1,2,3....,11

#Τα παρακάτω δεν ζητούνται αλλά τα πρόσθεσα για λόγους πληρότητας
print("\nΑναθέσεις πελατών (ποιον εξυπηρετεί ποιος):")
for i in warehouses:
    for j in distributors:
        if x[i, j].X > 1e-3:  # αν η ποσότητα είναι σημαντική (>0.001 τόνοι)
            print(f"Αποθήκη {i+1} → Πελάτης {j+1}: {x[i,j].X:.2f} τόνοι") #i+1 και j+1 για τον ίδιο λόγο (βλ. πάνω)
print(f"\nΣυνολικό κόστος: {m.ObjVal:.2f} χιλ. ευρώ") 

Ανοιχτές Αποθήκες:
 - Αποθήκη 1 (χωρητικότητα: 300)
 - Αποθήκη 5 (χωρητικότητα: 275)
 - Αποθήκη 8 (χωρητικότητα: 220)
 - Αποθήκη 9 (χωρητικότητα: 270)
 - Αποθήκη 12 (χωρητικότητα: 180)

Αναθέσεις πελατών (ποιον εξυπηρετεί ποιος):
Αποθήκη 1 → Πελάτης 1: 120.00 τόνοι
Αποθήκη 1 → Πελάτης 2: 5.00 τόνοι
Αποθήκη 1 → Πελάτης 3: 75.00 τόνοι
Αποθήκη 1 → Πελάτης 4: 100.00 τόνοι
Αποθήκη 5 → Πελάτης 10: 150.00 τόνοι
Αποθήκη 5 → Πελάτης 11: 5.00 τόνοι
Αποθήκη 5 → Πελάτης 12: 120.00 τόνοι
Αποθήκη 8 → Πελάτης 2: 75.00 τόνοι
Αποθήκη 8 → Πελάτης 5: 45.00 τόνοι
Αποθήκη 8 → Πελάτης 6: 100.00 τόνοι
Αποθήκη 9 → Πελάτης 5: 65.00 τόνοι
Αποθήκη 9 → Πελάτης 11: 90.00 τόνοι
Αποθήκη 12 → Πελάτης 7: 90.00 τόνοι
Αποθήκη 12 → Πελάτης 8: 60.00 τόνοι
Αποθήκη 12 → Πελάτης 9: 30.00 τόνοι

Συνολικό κόστος: 17929.23 χιλ. ευρώ
